In [9]:
from pathlib import Path
import pandas as pd
import sys, os

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "config.py").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))
from config import ROOT_DIR, OUTPUT_DIR

CSV_PATH = Path(OUTPUT_DIR) / "merged_panel.csv"
df = pd.read_csv(CSV_PATH)

# detect PE premium column (case-insensitive)
pe_candidates = [c for c in df.columns if "pe" in c.lower() and ("prem" in c.lower() or "premium" in c.lower())]
if not pe_candidates:
    pe_candidates = [c for c in df.columns if "premium" in c.lower() or c.lower().endswith("_pe")]
if not pe_candidates:
    raise ValueError(f"Could not find a PE premium column. Available columns: {list(df.columns)}")
pe_col = pe_candidates[0]

# coerce to numeric
df[pe_col] = pd.to_numeric(df[pe_col], errors="coerce")

# detect identity columns
ticker_candidates = [c for c in df.columns if c.lower() in ("ticker", "symbol", "permno", "gvkey", "cusip")]
ticker_col = ticker_candidates[0] if ticker_candidates else None
name_candidates = [c for c in df.columns if any(k in c.lower() for k in ("company", "firm", "name")) and c != ticker_col]
name_col = name_candidates[0] if name_candidates else None

# summary stats
mean_pe = df[pe_col].mean()
max_pe = df[pe_col].max()
min_pe = df[pe_col].min()

print(f"PE premium column: {pe_col}")
print(f"Mean PE premium: {mean_pe}")
print(f"Max PE premium: {max_pe}")
print(f"Min PE premium: {min_pe}")

if ticker_col:
    group_cols = [ticker_col]
    if name_col and name_col != ticker_col:
        group_cols.append(name_col)
    firm_pe = (
        df.dropna(subset=[pe_col])
        .groupby(group_cols, as_index=False)[pe_col]
        .mean()
        .rename(columns={pe_col: "avg_pe_premium"})
        .sort_values("avg_pe_premium", ascending=False)
    )
    top_10 = firm_pe.head(10)
    bottom_10 = firm_pe.tail(10).sort_values("avg_pe_premium", ascending=True)

    print("Top 10 firms by average PE premium:")
    print(top_10.to_string(index=False))

    print("\nBottom 10 firms by average PE premium:")
    print(bottom_10.to_string(index=False))
else:
    ranked = df.dropna(subset=[pe_col]).sort_values(pe_col, ascending=False)
    top_10 = ranked.head(10)[[pe_col]]
    bottom_10 = ranked.tail(10)[[pe_col]].sort_values(pe_col, ascending=True)

    print("Top 10 rows by PE premium:")
    print(top_10.to_string(index=False))

    print("\nBottom 10 rows by PE premium:")
    print(bottom_10.to_string(index=False))

PE premium column: pe_premium
Mean PE premium: 2.123196774261603
Max PE premium: 43.1201385
Min PE premium: -17.505487000000002
Top 10 firms by average PE premium:
ticker                name  avg_pe_premium
  ARES     Ares Management       42.934785
   TKO  TKO Group Holdings       40.626554
   APP            AppLovin       15.343509
     V           Visa Inc.       13.628941
  NVDA              Nvidia       13.127107
  NXPI  NXP Semiconductors       13.101696
   DHR Danaher Corporation       12.719398
   WDC     Western Digital       11.677423
  TRMB        Trimble Inc.       10.764627
  DXCM              Dexcom       10.199448

Bottom 10 firms by average PE premium:
ticker                      name  avg_pe_premium
    VZ                   Verizon      -17.505487
   FOX Fox Corporation (Class B)      -14.011165
  FOXA Fox Corporation (Class A)      -12.200640
   DVN              Devon Energy       -9.433996
    GD          General Dynamics       -7.481866
  META            Meta Platfo